## Phi-4 RAG ਨਾਲ Azure AI Search

ਇਹ ਨੋਟਬੁੱਕ ਦਿਖਾਉਂਦਾ ਹੈ ਕਿ ਕਿਵੇਂ Phi-4-mini ਅਤੇ Phi-4-multimodal ਨੂੰ Retrieval Augmented Generation (RAG) ਲਈ Azure AI Search ਨਾਲ ਵਰਤਿਆ ਜਾ ਸਕਦਾ ਹੈ। ਇਹ ਸਿਰਫ਼ ਇੱਕ ਮਾਡੈਲ (ਟੈਕਸਟ-ਕੇਵਲ) ਅਤੇ ਬਹੁ ਮਾਡੈਲ (ਟੈਕਸਟ ਅਤੇ ਚਿੱਤਰ) ਦੋਹਾਂ ਪੱਖਾਂ ਨੂੰ ਕਵਰ ਕਰਦਾ ਹੈ।

**ਜ਼ਰੂਰੀ ਸ਼ਰਤਾਂ:**
*   Azure AI Search ਵੈਕਟਰ ਇੰਡੈਕਸ (ਇੱਕ ਬਣਾਉਣ ਲਈ [ਇਹ ਨਿਰਦੇਸ਼](https://learn.microsoft.com/azure/search/search-get-started-portal-import-vectors?tabs=sample-data-storage%2Cmodel-aoai%2Cconnect-data-storage) ਫੋਲੋ ਕਰੋ)
*   Phi-4-mini ਜਾਂ Phi-4-multimodal ਐਂਡਪੋਇੰਟ Microsoft Foundry ਵਿੱਚ ਤੈਨਾਤ


In [ ]:
! pip install azure-ai-inference azure-search-documents

### ਟੈਕਸਟ-ਸਿਰਫ RAG ਫਿ-4-ਮਿਨੀ ਨਾਲ

ਇਸ ਸੈਕਸ਼ਨ ਵਿੱਚ ਦਿਖਾਇਆ ਗਿਆ ਹੈ ਕਿ ਕੇਵਲ ਟੈਕਸਟ ਨੂੰ ਇਨਪুট ਵਜੋਂ ਵਰਤਦੇ ਹੋਏ RAG ਲਈ ਚੈਟ ਕੰਪਲੀਸ਼ਨ ਮਾਡਲ ਵਜੋਂ ਫਿ-4-ਮਿਨੀ ਕਿਵੇਂ ਵਰਤਣਾ ਹੈ। ਇਸ ਵਿੱਚ Microsoft Foundry Inference ਅਤੇ Azure AI Search ਨਾਲ ਕਨੈਕਟ ਹੋਣਾ, ਪ੍ਰਸੰਗਿਕ ਦਸਤਾਵੇਜ਼ਾਂ ਨੂੰ ਪ੍ਰਾਪਤ ਕਰਨਾ ਅਤੇ ਪ੍ਰਾਪਤ ਕੀਤੇ ਸੰਦਰਭ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਜਵਾਬ ਤਿਆਰ ਕਰਨਾ ਸ਼ਾਮਿਲ ਹੈ।


In [ ]:
# Step 1: Connect to Microsoft Foundry Inference & Azure AI Search
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import UserMessage
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

chat_client = ChatCompletionsClient(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"], # Phi-4-mini endpoint
    credential=AzureKeyCredential(os.environ["AZURE_INFERENCE_CREDENTIAL"]),
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 10):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query,vector_queries=[vector_query],select=["content"],top=top)
    return [doc["content"] for doc in results]

# Step 3: Generate answer using RAG!
def generate_rag_response(query: str):
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    response = chat_client.complete(messages=[UserMessage(content=prompt)])
    return response.choices[0].message.content

# Example usage
user_query = "What is the capital of France?"
answer = generate_rag_response(user_query)
print(f"Q: {user_query}\nA: {answer}")


### Phi-4-multimodal ਨਾਲ ਬਹੁ-ਮੋਡਲ RAG

ਇਸ ਹਿੱਸੇ ਵਿੱਚ ਦਿਖਾਇਆ ਗਿਆ ਹੈ ਕਿ RAG ਲਈ ਇੱਕ ਚੈਟ ਪੂਰਨ ਮਾਡਲ ਵਜੋਂ Phi-4-multimodal ਦਾ ਕਿਵੇਂ ਇਸਤੇਮਾਲ ਕਰਨਾ ਹੈ, ਜਿਸ ਵਿੱਚ ਲਿਖਤੀ ਅਤੇ ਚਿੱਤਰ ਦੋਵਾਂ ਇਨਪੁਟ ਸ਼ਾਮਲ ਹਨ। ਇਹ ਅਜ਼ੁਰ ਏ.ਆਈ. ਇਨਫਰੰਸ ਅਤੇ ਅਜ਼ੁਰ ਏ.ਆਈ. ਸਿਰਚ ਨਾਲ ਜੁੜਨ, ਸਬੰਧਿਤ ਦਸਤਾਵੇਜ਼ ਪ੍ਰਾਪਤ ਕਰਨ ਅਤੇ ਇੱਕ ਬਹੁ-ਮੋਡਲ ਜਵਾਬ ਤਿਆਰ ਕਰਨ ਦਾ ਕਵਰੇਜ ਕਰਦਾ ਹੈ।

**ਨੋਟ:** ਜੇ ਤੁਹਾਡੀ ਅਜ਼ੁਰ ਏ.ਆਈ. ਸਿਰਚ ਇੰਡੈਕਸ ਵਿੱਚ `text_vector` ਅਤੇ `image_vector` ਦੋਵਾਂ ਫੀਲਡ ਹਨ, ਤਾਂ ਤੁਸੀਂ ਬਹੁ-ਵੇਕਟਰ ਕੈਵਰੀ ਵੀ ਕਰ ਸਕਦੇ ਹੋ।


In [ ]:
# Step 1: Connect to Azure AI Inference & Azure AI Search
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery


chat_client = ChatCompletionsClient(
    endpoint=os.environ["AZURE_INFERENCE_ENDPOINT"], #Phi-4-multimodal serverless endpoint
    credential=AzureKeyCredential(os.environ["AZURE_INFERENCE_CREDENTIAL"]),
)

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name=os.environ["AZURE_SEARCH_INDEX"],
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# Step 2: Retrieve relevant documents from Azure AI Search
def retrieve_documents(query: str, top: int = 3):
    vector_query = VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="text_vector")
    results = search_client.search(search_text=query, vector_queries=[vector_query], select=["content"], top=top)    
    return [doc["content"] for doc in results]

## Example for Muli-modal search if you have a text_vector AND image_vector field in your vector_index
## NOTE, image vectorization is in preview at the time of writing this code, please use azure-search-documents pypi version >11.6.0b6 
# def retrieve_documents_multimodal(query: str, image_url: str, top: int = 3):
#     text_vector_query = VectorizableTextQuery(
#         text=query,
#         k_nearest_neighbors=top,
#         fields="text_vector",
#         weight=0.5  # Adjust weight as needed
#     )
#     image_vector_query = VectorizableImageUrlQuery(
#         url=image_url,
#         k_nearest_neighbors=top,
#         fields="image_vector",
#         weight=0.5  # Adjust weight as needed
#     )

#     results = search_client.search(
#         search_text=query,  
#         vector_queries=[text_vector_query, image_vector_query],
#         select=["content"],
#         top=top
#     )
#     return [doc["content"] for doc in results]


# Step 3: Generate a multimodal RAG-based answer using retrieved text and an image input
def generate_multimodal_rag_response(query: str, image_url: str):
    # Retrieve text context from search
    docs = retrieve_documents(query)
    context = "\n---\n".join(docs)

    # Build a prompt that combines the retrieved context with the user query
    prompt = f"""You are a helpful assistant. Use only the following context to answer the question. If the answer isn't in the context, say 'I don't know'.
    Context: {context} Question: {query} Answer:"""
    # Create a chat request that includes both text and image input
    response = chat_client.complete(
        messages=[
            SystemMessage(content="You are a helpful assistant that can process both text and images."),
            UserMessage(
                content=[
                    TextContentItem(text=prompt),
                    ImageContentItem(image_url=ImageUrl(url=image_url, detail=ImageDetailLevel.HIGH)),
                ]
            ),
        ]
    )
    return response.choices[0].message.content

# Example usage
user_query = "Can you search for similar items to this shoe?"
sample_image_url = "https://images.unsplash.com/photo-1542291026-7eec264c27ff?q=80&w=1770&auto=format&fit=crop&ixlib=rb-4.0.3"
answer = generate_multimodal_rag_response(user_query, sample_image_url)
print(f"Q: {user_query}\nA: {answer}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰ ਕੀਤਾ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਪ੍ਰਯਾਸ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਜ਼ਾਹਰ ਕਰੋ ਕਿ ਸਵਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਪਸ਼ਟਤਾਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਅਧਿਕਾਰਿਤ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਨਾਲ ਪੈਦਾ ਹੋਣ ਵਾਲੀ ਕਿਸੇ ਵੀ ਗਲਤਫਹਮੀ ਜਾਂ ਗਲਤ ਵਿਵੇਚਨਾ ਲਈ ਅਸੀਂ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
